# Claude API を呼び出す

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm_api/claude_basic.ipynb)

NLP クックブック 14-07 ／ LLM の API（Gemini と Claude）

> API キーが必要（Colab 左の 🔑 シークレットに登録）

Claude を API 経由で使いたい（有料。無料で試すなら前の Gemini のレシピ）。Colab のシークレット機能で API キーを安全に扱いたい。

In [ ]:
!pip install -q anthropic

In [ ]:
import anthropic
from google.colab import userdata

# 左の 🔑 (シークレット) に ANTHROPIC_API_KEY を登録し、「ノートブックからのアクセス」を有効にしておく
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

response = client.beta.messages.create(
    model="claude-opus-5-5",
    max_tokens=16000,
    system="あなたは自然言語処理の専門家です。簡潔に答えてください。",
    messages=[{"role": "user", "content": "BPE と WordPiece の違いを 3 行で説明してください。"}],
    betas=["server-side-fallback-2026-07-01"],
    fallbacks="default",   # 安全フィルタで応答が拒否された場合に、別のモデルで自動的に再実行する
)

if response.stop_reason == "refusal":
    print("応答が拒否されました:", response.stop_details)
else:
    print("".join(block.text for block in response.content if block.type == "text"))
print(response.usage.input_tokens, "入力トークン /", response.usage.output_tokens, "出力トークン")

In [ ]:
# 対話を続ける: API は状態を持たないので、履歴をすべて送り直す
history = [{"role": "user", "content": "「は」と「が」の違いを一言で。"}]
r1 = client.messages.create(model="claude-opus-5-5", max_tokens=16000, messages=history)
history += [{"role": "assistant", "content": r1.content}, {"role": "user", "content": "例文を 1 つずつ挙げてください。"}]
r2 = client.messages.create(model="claude-opus-5-5", max_tokens=16000, messages=history)
print("".join(b.text for b in r2.content if b.type == "text"))

- API キーをノートブックに直接書かない。共有や GitHub への push で漏れる。
- `response.content` はブロックのリスト（テキストのほかに思考ブロックなどが入る）。`type == "text"` のものだけを取り出す。
- `max_tokens` は出力の上限。小さくしすぎると途中で切れる (`stop_reason == "max_tokens"`)。
- 履歴に応答を積むときは、テキストだけでなく `response.content` をそのまま渡す。
- モデル ID と料金は [公式ドキュメント](https://platform.claude.com/docs/en/about-claude/models/overview) で確認する。大量の単純な処理には、より安価な `claude-haiku-5-5` なども選べる。